In [ ]:
# MATLAB section 1/2 for DecodingExampleWithHist: 1-D Stimulus Decode with History Effect

# % 1-D Stimulus Decode with History Effect
# In the above decoding example, the simulated neurons did not have memory.
# That is their previous firing activity did not modulate their current
# probability of firing. In reality the firing history does affect the
# probabilty of neuronal firing (eg. refractory period, bursting, etc.).
# In this example, we simulate a population a neurons that exhibit this
# type of history dependence. We then decode the stimulus activity based on
# a conditional intensity function that includes the correct history terms
# and one that assumes no history dependence.
#
#
# MATLAB: close all;
# clear all;
# MATLAB: delta = 0.001; Tmax = 1;
# MATLAB: time = 0:delta:Tmax;
# MATLAB: f=1; b1=1;b0=-2;
# MATLAB: stimData = b1*sin(2*pi*f*time);
# MATLAB: e = zeros(length(time),1);   %No Ensemble input
# MATLAB: mu = b0; %baseline firing rate
# MATLAB: Ts=delta;
#
# MATLAB: histCoeffs= [-2 -2 -4];
# MATLAB: windowTimes=[0 .001 0.002 0.003];
# MATLAB: histObj = History(windowTimes);
# MATLAB: filts = histObj.toFilter(Ts); %Convert to transfer function matrix
# MATLAB: H=histCoeffs*filts; %scale each window transfer function by its coefficient
# MATLAB: S=tf([1],1,Ts,'Variable','z^-1'); %Feed the stimulus in directly
# MATLAB: E=tf([0],1,Ts,'Variable','z^-1'); %No ensemble effect
# MATLAB: stim=Covariate(time',stimData,'Stimulus','time','s','Voltage',{'sin'});
# MATLAB: ens =Covariate(time',e,'Ensemble','time','s','Spikes',{'n1'});
# MATLAB: numRealizations = 20;    %Number of sample paths to generate
# MATLAB: sC=CIF.simulateCIF(mu,H,S,E,stim,ens,numRealizations);
#
# MATLAB: figure;
# MATLAB: subplot(2,1,1); sC.plot;
# MATLAB: subplot(2,1,2); stim.plot;
#
#
# MATLAB: for i=1:numRealizations
# Construct a CIF object for each realization based on our encoding
# results above
# correct CIF w/ History
# MATLAB:     lambdaCIF{i} = CIF([mu b1],{'1','x'},{'x'},'binomial',histCoeffs,histObj);
# CIF ignoring the history effect
# MATLAB:     lambdaCIFNoHist{i} = CIF([mu b1],{'1','x'},{'x'},'binomial');
# MATLAB: end
#
#
#
#
# MATLAB: sC.resample(1/delta);
# MATLAB: dN=sC.dataToMatrix;
# Make noise according to the dynamic range of the stimulus
# MATLAB: Q=2*std(stim.data(2:end)-stim.data(1:end-1));
# MATLAB: Px0=.1; A=1;
# Decode with the correct and incorrect CIFs
#
# MATLAB: [x_p, W_p, x_u, W_u] = DecodingAlgorithms.PPDecodeFilter(A, Q, Px0, dN',lambdaCIF,delta);
# MATLAB: [x_pNoHist, W_pNoHist, x_uNoHist, W_uNoHist] = DecodingAlgorithms.PPDecodeFilter(A, Q, Px0, dN',lambdaCIFNoHist,delta);
#
#
# Compare the results
# MATLAB: figure;
# MATLAB: subplot(2,1,1);
# MATLAB: zVal=3;
# MATLAB: ciLower = min(x_u(1:end)-zVal*squeeze(W_u(1:end))',x_u(1:end)+zVal*squeeze(W_u(1:end))');
# MATLAB: ciUpper = max(x_u(1:end)-zVal*squeeze(W_u(1:end))',x_u(1:end)+zVal*squeeze(W_u(1:end))');
# MATLAB: hEst=plot(time,x_u(1:end),'b',time,ciLower,'g',time,ciUpper,'r'); hold on;
# MATLAB: hold all;
#
# MATLAB: hStim=stim.plot([],{{' ''k'',''Linewidth'',2'}});
# MATLAB: legend off;
# MATLAB: legend([hEst(1) hEst(2) hEst(3) hStim],'x_{k|k}(t)',strcat('x_{k|k}(t)-',num2str(zVal),'\sigma_{k|k}'),...
# MATLAB:         strcat('x_{k|k}(t)+',num2str(zVal),'\sigma_{k|k}'),'x_{k|k}(t)','x(t)');
# MATLAB: title(['Decoded Stimulus +/- 99% confidence intervals using ' num2str(numRealizations) ' cells']);
#
# MATLAB: subplot(2,1,2);
# MATLAB: zVal=3;
# MATLAB: ciLower = min(x_uNoHist(1:end)-zVal*squeeze(W_uNoHist(1:end))',x_uNoHist(1:end)+zVal*squeeze(W_uNoHist(1:end))');
# MATLAB: ciUpper = max(x_uNoHist(1:end)-zVal*squeeze(W_uNoHist(1:end))',x_uNoHist(1:end)+zVal*squeeze(W_uNoHist(1:end))');
# MATLAB: hEst=plot(time,x_uNoHist(1:end),'b',time,ciLower,'g',time,ciUpper,'r'); hold on;
# MATLAB: hold all;
#
# MATLAB: hStim=stim.plot([],{{' ''k'',''Linewidth'',2'}});
# MATLAB: legend off;
# MATLAB: legend([hEst(1) hEst(2) hEst(3) hStim],'x_{k|k}(t)',strcat('x_{k|k}(t)-',num2str(zVal),'\sigma_{k|k}'),...
# MATLAB:         strcat('x_{k|k}(t)+',num2str(zVal),'\sigma_{k|k}'),'x_{k|k}(t)','x(t)');
# MATLAB: title(['Decoded Stimulus No Hist +/- 99% confidence intervals using ' num2str(numRealizations) ' cells']);
#

# Python translation bootstrap for this helpfile.

# Topic: DecodingExampleWithHist
# Execution group: smoke
# Workflow family: decoding_1d
# Paper DOI: 10.1016/j.jneumeth.2012.08.009
# PMID: 22981419
# Help page: docs/help/examples/DecodingExampleWithHist.md


import matplotlib
matplotlib.use("Agg")
try:
    from matplotlib_inline.backend_inline import set_matplotlib_formats
    matplotlib.use("module://matplotlib_inline.backend_inline")
    set_matplotlib_formats("png")
except Exception:
    pass

import numpy as np
import matplotlib.pyplot as plt

from nstat.analysis import Analysis
from nstat.cif import CIFModel
from nstat.decoding import DecodingAlgorithms
from nstat.events import Events
from nstat.history import HistoryBasis
from nstat.signal import Covariate
from nstat.spikes import SpikeTrain, SpikeTrainCollection
from nstat.trial import CovariateCollection, Trial, TrialConfig

TOPIC = "DecodingExampleWithHist"
FAMILY = "decoding_1d"
np.random.seed(2026)
rng = np.random.default_rng(2026)
print(f"Running notebook topic: {TOPIC} (family={FAMILY})")

if "MATLAB_LINE_TRACE" not in globals():
    MATLAB_LINE_TRACE = []

def matlab_line(line: str):
    MATLAB_LINE_TRACE.append(line)
    return line

def validate_numeric_checkpoints(metrics: dict[str, float], limits: dict[str, tuple[float, float]], topic: str) -> None:
    if not metrics:
        raise AssertionError(f"DecodingExampleWithHist: CHECKPOINT_METRICS is empty")
    for key, value in metrics.items():
        if not np.isfinite(value):
            raise AssertionError(f"DecodingExampleWithHist: metric '{key}' is not finite: {value}")
    for key, (lo, hi) in limits.items():
        if key not in metrics:
            raise AssertionError(f"DecodingExampleWithHist: missing checkpoint metric '{key}'")
        value = float(metrics[key])
        if value < float(lo) or value > float(hi):
            raise AssertionError(
                f"DecodingExampleWithHist: metric '{key}'={value:.6f} outside [{float(lo):.6f}, {float(hi):.6f}]"
            )
    print(f"Numeric checkpoints for {topic}:", metrics)


In [ ]:
# MATLAB section 2/2 for DecodingExampleWithHist: Section

# %
# We see that inclusion of history effect improves (as expected) the
# decoding of the stimulus based on the point process observations

# Python translation execution block for this helpfile.

# 1D Decoding workflow: decode latent state sequence from population spikes.
n_units = 14
n_states = 17
n_time = 260
state_idx = np.arange(n_states)

transition = np.zeros((n_states, n_states), dtype=float)
for i in range(n_states):
    for j, w in ((i - 1, 0.2), (i, 0.6), (i + 1, 0.2)):
        if 0 <= j < n_states:
            transition[i, j] += w
    transition[i, :] /= np.sum(transition[i, :])

latent = np.zeros(n_time, dtype=int)
latent[0] = n_states // 2
for t in range(1, n_time):
    latent[t] = rng.choice(n_states, p=transition[latent[t - 1]])

centers = np.linspace(0.0, n_states - 1, n_units)
widths = np.full(n_units, 2.1)
state_axis = np.arange(n_states)[None, :]
tuning = 0.06 + 0.42 * np.exp(-0.5 * ((state_axis - centers[:, None]) / widths[:, None]) ** 2)

use_history = TOPIC in {"DecodingExampleWithHist", "nSTATPaperExamples"}

if use_history:
    gain = np.ones(n_time, dtype=float)
    counts = np.zeros((n_units, n_time), dtype=float)
    prev = 0.0
    for t in range(n_time):
        gain[t] = np.exp(0.50 * prev)
        lam = tuning[:, latent[t]] * gain[t]
        counts[:, t] = rng.poisson(lam)
        prev = float(np.mean(counts[:, t]))

    decoded_raw, _ = DecodingAlgorithms.decode_state_posterior(counts, tuning, transition)
    corrected = counts / gain[None, :]
    decoded, posterior = DecodingAlgorithms.decode_state_posterior(corrected, tuning, transition)
    rmse_raw = float(np.sqrt(np.mean((decoded_raw - latent) ** 2)) / (n_states - 1))
    rmse_dec = float(np.sqrt(np.mean((decoded - latent) ** 2)) / (n_states - 1))
else:
    counts = np.zeros((n_units, n_time), dtype=float)
    for t in range(n_time):
        counts[:, t] = rng.poisson(tuning[:, latent[t]])
    decoded, posterior = DecodingAlgorithms.decode_state_posterior(counts, tuning, transition)
    rmse_raw = float(np.sqrt(np.mean((decoded - latent) ** 2)) / (n_states - 1))
    rmse_dec = rmse_raw

fig, axes = plt.subplots(2, 1, figsize=(9, 7), sharex=True)
axes[0].plot(latent, label="true", linewidth=1.2)
axes[0].plot(decoded, label="decoded", linewidth=1.0)
axes[0].set_title(f"{TOPIC}: latent-state decoding")
axes[0].set_ylabel("state")
axes[0].legend(loc="upper right")

im = axes[1].imshow(posterior, aspect="auto", origin="lower", cmap="viridis")
axes[1].set_title("Posterior over latent states")
axes[1].set_xlabel("time bin")
axes[1].set_ylabel("state")
fig.colorbar(im, ax=axes[1], fraction=0.03, pad=0.02)

plt.tight_layout()
plt.show()

print("rmse_raw", rmse_raw, "rmse_final", rmse_dec)
assert np.max(np.abs(np.sum(posterior, axis=0) - 1.0)) < 1e-6
if use_history:
    assert rmse_dec <= rmse_raw + 0.03

CHECKPOINT_METRICS = {
    "rmse_raw": float(rmse_raw),
    "rmse_dec": float(rmse_dec),
}
CHECKPOINT_LIMITS = {
    "rmse_raw": (0.0, 0.65),
    "rmse_dec": (0.0, 0.65),
}


# Execution checkpoints used by CI.
assert TOPIC != "", "Missing topic metadata"
validate_numeric_checkpoints(CHECKPOINT_METRICS, CHECKPOINT_LIMITS, TOPIC)
print("Topic-specific checkpoint for", TOPIC)
print("Notebook checkpoints passed for", TOPIC)
